# Classificação de diabetes com XGBoost (dataset NIDDK)

<mark>Este notebook é didático e não deve ser utilizado para realizar diagnósticos</mark>

Este notebook utiliza os dados preparados por `01-prepare-niddk-diabetes.ipynb` para treinar e avaliar um único classificador binário: **XGBoost**. O fluxo é intencionalmente compacto: carregar os dados, definir `X` e `y`, separar os conjuntos, treinar o modelo e avaliar suas previsões.

## Importações e caminhos do projeto

### Bibliotecas

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

sns.set_theme(style="whitegrid")


def find_project_root() -> Path:
    """Retorna a raiz do repositório quando o notebook é executado neste projeto."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "ml" / "pyproject.toml").is_file() and (
            candidate / "README.md"
        ).is_file():
            return candidate
    raise FileNotFoundError("Abra este notebook dentro do repositório do projeto.")

### Variáveis

In [ ]:
PROJECT_ROOT = find_project_root()

DATA_PATH = PROJECT_ROOT / "data" / "niddk-diabetes" / "diabetes.parquet"
TARGET_COLUMN = "diabetes"
IDENTIFIER_COLUMN = "id_paciente"

RANDOM_STATE = 42

## Carregar o dataset preparado

O notebook de preparação traduz as colunas de origem para português e as salva em Parquet. Antes de prosseguir, verificamos se as colunas de alvo e identificador esperadas estão presentes.

In [ ]:
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset preparado não encontrado em {DATA_PATH}. "
        "Execute primeiro 01-prepare-niddk-diabetes.ipynb."
    )

data = pd.read_parquet(DATA_PATH)
required_columns = {TARGET_COLUMN, IDENTIFIER_COLUMN}
missing_columns = required_columns.difference(data.columns)
if missing_columns:
    raise ValueError(f"O dataset não possui as colunas obrigatórias: {sorted(missing_columns)}")

print(f"Dimensões do dataset: {data.shape}")

In [ ]:
print(f"Distribuição das classes:\n{data[TARGET_COLUMN].value_counts(normalize=True).sort_index()}")

In [ ]:
display(data.head())

## Separar os conjuntos em variáveis de entrada `X` e alvo `y`

`diabetes` é o desfecho binário a ser previsto. `id_paciente` é removida porque identifica um registro em vez de descrever uma característica da pessoa, as demais colunas formam `X`.

In [ ]:
feature_columns = [
    column for column in data.columns if column not in {TARGET_COLUMN, IDENTIFIER_COLUMN}
]

X = data[feature_columns]
y = data[TARGET_COLUMN]

print(f"Dimensões de X: {X.shape}")
print(f"Dimensões de y: {y.shape}")
print(f"Variáveis de entrada: {feature_columns}")

## Criar os conjuntos de treino e teste

O conjunto de teste fica reservado até a avaliação. `stratify=y` preserva a proporção das classes em ambas as divisões, e `random_state` torna o resultado reproduzível.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f"Conjunto de treino: {X_train.shape[0]} linhas")
print(f"Conjunto de teste:  {X_test.shape[0]} linhas")
print(f"Variáveis:          {X_train.shape[1]}")

## Treinar um classificador XGBoost

O XGBoost constrói um conjunto de árvores de decisão sequencialmente; cada nova árvore se concentra nos erros das árvores anteriores. Modelos baseados em árvores não exigem padronização das variáveis, portanto nenhum escalonador é usado aqui. Os parâmetros abaixo são explícitos e moderados para uma linha de base reproduzível.

In [ ]:
model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

model.fit(X_train, y_train)

print("Treinamento do XGBoost concluído.")

## Avaliar no conjunto de testes

As probabilidades previstas são usadas para o ROC-AUC, enquanto as classes usam o limiar padrão de 0,5 do modelo. Precisão, revocação e F1 complementam a acurácia porque distinguem diferentes tipos de erro de classificação.

In [ ]:
y_pred = model.predict(X_test)
y_probability = model.predict_proba(X_test)[:, 1]

metrics = pd.Series(
    {
        "Acurácia": accuracy_score(y_test, y_pred),
        "Precisão": precision_score(y_test, y_pred, zero_division=0),
        "Revocação": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_probability),
    }
)

In [ ]:
display(metrics.to_frame("métrica").round(3))

In [ ]:
print(classification_report(y_test, y_pred, digits=3, zero_division=0))

## Examinar erros e a importância das variáveis

A matriz de confusão compara as classes previstas com as reais. A importância das variáveis é um resumo específico do modelo sobre a frequência e a eficácia com que uma variável foi usada nas árvores aprendidas; ela não estabelece causalidade ou relevância clínica.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=["Sem diabetes", "Diabetes"],
    cmap="Blues",
    colorbar=False,
    ax=ax,
)

ax.set_title("Matriz de confusão do XGBoost")
plt.show()

In [ ]:
feature_importance = (
    pd.Series(model.feature_importances_, index=feature_columns)
    .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(8, 5))
feature_importance.plot.barh(ax=ax, color="steelblue")

ax.set_title("Importância das variáveis no XGBoost")
ax.set_xlabel("Importância")
ax.set_ylabel("Variável")

plt.tight_layout()
plt.show()